# Compact MobileNetV4 — A2 distillation pilot
Latest revision: **COMPACT-A2-SELECTIVE-KD-2026-10-11-r1**

Keep the same 1.48M-parameter compact CNN. Start from its selected GT-only epoch120 checkpoint (Vehicle/Pedestrian 3D AP 2.89/0.78). Compare two 10-epoch continuations: GT-only control and GT + selective A2 geometry guidance. Both use identical seed, data augmentation, learning rate, frozen BatchNorm statistics, initialization and evaluation. No new inference layers, feature adapters or depth head are added.

The teacher is the already validated A2 ONNX model from M68; CPU caching does **not** rebuild MonoDETR, replace Torch, install Core ML, restore an old virtual environment or compile CUDA extensions. The student trains on Colab GPU. No MonoDGP is involved.

A train-only audit decides whether each class benefits from keypoint, dimension, orientation or depth targets. Labels remain the primary supervision. This is object-matched regression distillation, not a soft-classification/temperature experiment. If no signal qualifies, return the audit ZIP instead of forcing training. The audit does not claim that KD will improve validation accuracy.

Run sections **1–7 in order**. Keep the completed GT run unchanged. After a disconnect, rerun from the top: audited image records and completed epochs are resumable. Change RUN_ID after code/settings changes; never delete older results to bypass checks. Offline teacher caching runs once over 3,712 training frames and may take a while on Colab CPU/Drive. Full validation evaluates all 3,769 frames at epochs5 and10. The final ZIP excludes large checkpoints, teacher files and per-image predictions.

## 1. Mount Drive and define every helper
Set the three artifact paths here. The default teacher folder is the completed M68 r3 export; a downloaded M68 results ZIP is also accepted. No cells from an older notebook are required.

In [ ]:
# COMPACT-A2-SELECTIVE-KD-2026-10-11-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, os, shlex, subprocess, sys

REVISION = 'COMPACT-A2-SELECTIVE-KD-2026-10-11-r1'
RUN_ID = 'compact_a2_selective_kd_r1'
RUN_PILOT = True
MOBILE_REPO = Path('/content/mobile_adas3d')
BASELINE = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/compact_mnv4_gt/compact_mnv4_gt_r1')
TEACHER_EXPORT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_m68_onnx_cpu/m68_a2_onnx_cpu_r3/export')
DATASET = Path('/content/kitti_compact_mnv4')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/compact_a2_kd') / RUN_ID
SCRIPT = MOBILE_REPO/'scripts/distill_compact_a2.py'
PYTHON = Path(sys.executable)
LOGS = OUTPUT/'colab_logs'
LOGS.mkdir(parents=True, exist_ok=True)
READY = False

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    path = LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+', shlex.join(command), '\nDurable log:', path, flush=True)
    tail = deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n'); log.flush()
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally: process.stdout.close()
        status = process.wait(); log.write(f'\nEXIT CODE: {status}\n'); log.flush()
    if status: raise RuntimeError(f'Exit {status}; full log={path}\n'+'\n'.join(tail))
    return path

def stage(name, *extra):
    if not READY: raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON,'-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra], name, MOBILE_REPO)

print(REVISION, '\nOriginal GT run:', BASELINE, '\nNew output:', OUTPUT)


## 2. Update clean main, install ordinary dependencies and restore data
This does not change the completed run, install Torch/CUDA, or download a new teacher checkpoint.

In [ ]:
READY = False
if not MOBILE_REPO.exists():
    run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO], 'clone')
dirty = subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty: raise RuntimeError('Preserve tracked project edits; use a clean main checkout.\n'+dirty)
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'], 'update', MOBILE_REPO)
for required in (SCRIPT,MOBILE_REPO/'scripts/compact_a2_teacher.py',MOBILE_REPO/'models/compact_distillation.py',
                 BASELINE/'training_manifest.json',BASELINE/'selection.json',BASELINE/'checkpoints/epoch_120.pt'):
    if not required.is_file(): raise FileNotFoundError(required)
if not TEACHER_EXPORT.exists():
    raise FileNotFoundError(f'{TEACHER_EXPORT}: set TEACHER_EXPORT in section1 to the passed M68 export folder or results ZIP')
if OUTPUT.resolve().is_relative_to(BASELINE.resolve()): raise RuntimeError('Use a separate output, never the completed GT directory')
run_logged([PYTHON,'-m','pip','install','-q','timm==1.0.27','pyyaml','opencv-python-headless','pillow==11.3.0','onnxruntime==1.31.0'], 'dependencies')
run_logged([PYTHON,'-c',"import torch,timm,onnxruntime; assert torch.cuda.is_available(); assert timm.__version__=='1.0.27'; print(torch.__version__,torch.cuda.get_device_name(0),onnxruntime.__version__)"], 'gpu')
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in ('/content/drive/MyDrive/datasets/kitti','/content/kitti','/content/kitti_compact_mnv4'):
    command += ['--candidate',candidate]
run_logged(command,'restore_data',MOBILE_REPO)
READY = True


## 3. Tests, frozen inputs and baseline diagnostics
Verify the exact A2 ONNX model, passed decoded full-validation report and all16 original CPU fixtures. Runtime checks use the exact saved inputs and unchanged M68 output tolerances. Reconstructed images may differ only in sparse single-RGB-level PIL rounding; decoded guidance must still agree within0.05 pixels,0.01 metres and0.001 radians. Inspect 2D matching versus 3D geometry on the original compact validation predictions. This fixed-threshold diagnostic is **not 2D AP**, and validation results do not choose KD targets.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_compact_a2_distillation.py','-v'],'kd_tests',MOBILE_REPO)
stage('prepare','--baseline',BASELINE,'--teacher-export',TEACHER_EXPORT,'--dataset-root',DATASET,'--split-dir',SPLITS)
stage('diagnose')


## 4. Resume-safe teacher cache and train-only geometry audit
A2 runs on CPU, the frozen compact baseline on GPU. Targets are class-aware, one-to-one matches with 2D IoU ≥0.5 and teacher class confidence ≥0.1. Each class/component needs at least16 matches, at least10% lower mean error and lower median error. Each individual KD target must also be better than the frozen student. Exact-GT projection/solve roundtrips check decoder numerics. There are no validation frames in this cache.

All input files are hashed when first cached. Resume uses size/mtime checks and rechecks bytes when metadata changes, avoiding repeated full-image reads from Drive. Completed cache records are never silently replaced.

In [ ]:
stage('cache-audit')
AUDIT = json.loads((OUTPUT/'teacher_audit.json').read_text())
print(json.dumps(AUDIT,indent=2))
if not AUDIT['pilot_authorized']: raise RuntimeError('Return section7 bundle; no qualified teacher signal or failed geometry oracle')


## 5. Disposable GPU smoke for both recipes
Two updates each, finite loss/gradients required. These disposable weights never initialize either continuation. BatchNorm running statistics remain frozen in both roles.

In [ ]:
stage('smoke', '--role', 'control')
stage('smoke', '--role', 'kd')
for role in ('control','kd'):
    smoke = json.loads((OUTPUT/f'smoke_{role}.json').read_text())
    assert smoke['finite_gradients'] and smoke['optimizer_steps']==2


## 6. One matched GT continuation and one KD continuation
Each recipe trains10 epochs from original compact epoch120 weights, fresh AdamW at2e-5, batch4, seed123, the same natural class distribution and augmentation. GT losses stay at weight1. Only KD adds qualified training guidance, ramped over2 epochs. We are not mixing in class-rebalancing, architecture or temperature sweeps.

Epoch-boundary resume is automatic. Evaluate both on all3,769 validation frames after epochs5 and10. The report distinguishes improvement from simple continuation versus improvement attributable to KD. A useful pilot is not deployment qualification; its trained export still needs a phone retest.

In [ ]:
if RUN_PILOT:
    stage('train', '--role', 'control')
    stage('train', '--role', 'kd')
    print(json.dumps(json.loads((OUTPUT/'comparison.json').read_text()),indent=2))
else:
    print('Pilot disabled; return audit and smoke results first.')


## 7. Return a small review bundle
This section can also be run after an audit/smoke failure. Checkpoints remain on Drive. Send compact_a2_kd_results.zip before launching longer training.

In [ ]:
stage('bundle')
BUNDLE = OUTPUT/'compact_a2_kd_results.zip'
print('Results:',BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
